In [1]:
from pyspark.sql.functions import col, isnan

In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
        .appName("search_tb") \
        .getOrCreate()

bash: /opt/conda/miniconda3/lib/libtinfo.so.6: no version information available (required by bash)
bash: /opt/conda/miniconda3/lib/libtinfo.so.6: no version information available (required by bash)
26/04/24 09:20:39 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/04/24 09:20:39 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/24 09:20:40 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Pleas

In [3]:
date = "2026-03-07"
path_to_interactions_data = f"gs://data-science-prod-redshift-archive/spectrum/xstream_users_engagement/day={date}"
interactions_data = spark.read.parquet(path_to_interactions_data)

26/04/24 09:21:05 WARN YarnScheduler: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/04/24 09:21:20 WARN YarnScheduler: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/04/24 09:21:35 WARN YarnScheduler: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/04/24 09:21:50 WARN YarnScheduler: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/04/24 09:22:05 WARN YarnScheduler: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/04/24 09:22:20 WARN YarnScheduler: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registere

# OLD codes

In [3]:
from pyspark.sql.functions import col, when, lit, lower

interactions_data = interactions_data.select("day", "events_ts", "events_meta_action", "events_meta_source_name",\
            "events_av", "events_dt", "events_os", "events_meta_search_session_id", "events_uid", "content_id",\
            "events_appid", "events_meta_keyword", "events_meta_asset_name", "events_meta_content_name", "eventtype")\
            .filter(col("events_meta_search_session_id").isNotNull())\
            .distinct()
path_to_user_mapping = f"gs://data-science-prod-redshift-archive/spectrum/device_user_mapping_optimised/day={date}"
user_mapping_data = spark.read.parquet(path_to_user_mapping).filter(col("temp_uid").isNotNull())\
                    .select("temp_uid", "reg_uid", "day")\
                    .withColumnRenamed("day", "um_day").distinct()
interactions_data1 = interactions_data.join(user_mapping_data,\
                                            on=((user_mapping_data.temp_uid == interactions_data.events_uid) & (user_mapping_data.um_day == interactions_data.day)),\
                                            how="left")

interactions_data1 = interactions_data1.withColumn("uid", when(col("reg_uid").isNull(), col("events_uid")).otherwise(col("reg_uid")))\
                                    .select("day", "events_meta_action","events_ts", "events_meta_source_name",\
                                            "events_av", "events_dt", "events_os", "events_meta_search_session_id", "uid", "content_id",\
                                            "events_appid", "events_meta_keyword", "events_meta_asset_name", "events_meta_content_name", "eventtype")\
                                    .filter((col("events_meta_action").isin(['search_input','text_input', 'voice_search',\
                                                                             'search_click', 'voice search', "search_result_consumed", "tab_click"\
                                                                            'tile_click','banner_cta_click', 'item_search','autosuggest_click'])))

interactions_data1 = interactions_data1.withColumn("consumption",\
                                                   when(((col("events_meta_action").isin(['tile_click','banner_cta_click'])) &\
                                                         col("events_meta_source_name").isin(['search_page','searchLayout']) ) |\
                                                        (col("events_meta_action") == "search_result_consumed"), lit(1))\
                                                   .otherwise(lit(0)))\
                                        .withColumn("session", when(col("events_meta_action")\
                                                        .isin('search_input', 'text_input', 'voice_search', 'search_click', 'voice search') |\
                                                       ((col("events_meta_action") == "tab_click") & (lower("events_meta_asset_name") == 'search') & (lower("events_appid") == 'largescreen'))\
                                                       , lit(1)).otherwise(lit(0))
                                                   )\
                                        .withColumn("autosuggest", when(col("events_meta_action").isin(['item_search','autosuggest_click']) &\
                                                                    col("events_meta_asset_name").isin(['autosuggest','auto_suggestion']) &\
                                                                    (col("eventtype") == "click"), lit(1)).otherwise(lit(0)))
interactions_data1.select("events_meta_search_session_id", "events_ts", "events_meta_keyword", "events_meta_action",\
                          "events_meta_content_name", "content_id", "events_meta_asset_name", "consumption", "session", "autosuggest")\
                    .orderBy(col("events_meta_search_session_id").asc(), col("events_ts").asc()).show(truncate = False, vertical = True)

-RECORD 0-------------------------------------------------------------
 events_meta_search_session_id | 0000023E-6AB9-438A-A5C5-24719B03DC8F 
 events_ts                     | 1772889594842                        
 events_meta_keyword           | NULL                                 
 events_meta_action            | search_input                         
 events_meta_content_name      | NULL                                 
 content_id                    | NULL                                 
 events_meta_asset_name        | search_tab                           
 consumption                   | 0                                    
 session                       | 1                                    
 autosuggest                   | 0                                    
-RECORD 1-------------------------------------------------------------
 events_meta_search_session_id | 0000023E-6AB9-438A-A5C5-24719B03DC8F 
 events_ts                     | 1772889596897                        
 event

In [5]:
interactions_data1.select("events_meta_search_session_id", "events_ts", "events_meta_keyword", "events_meta_action",\
                          "events_meta_content_name", "content_id", "events_meta_asset_name", "consumption", "session", "autosuggest")\
                    .filter(col("autosuggest") == 1)\
                    .orderBy(col("events_meta_search_session_id").asc(), col("events_ts").asc()).show(truncate = False, vertical = True)

-RECORD 0---------------------------------------------------------------------------
 events_meta_search_session_id | 000007a8-26fc-4def-ae27-378a1707aac3-1772886619240 
 events_ts                     | 1772886624172                                      
 events_meta_keyword           | maru                                               
 events_meta_action            | item_search                                        
 events_meta_content_name      | Maruthi Nagar Subrahmanyam                         
 content_id                    | NULL                                               
 events_meta_asset_name        | autosuggest                                        
 consumption                   | 0                                                  
 session                       | 0                                                  
 autosuggest                   | 1                                                  
-RECORD 1--------------------------------------------------------

In [12]:
interactions_data1.select("events_meta_search_session_id", "events_ts", "events_meta_keyword", "events_meta_action",\
                          "events_meta_content_name", "content_id", "events_meta_asset_name", "consumption", "session", "autosuggest")\
                    .filter(col("consumption") == 1)\
                    .orderBy(col("events_meta_search_session_id").asc(), col("events_ts").asc()).show(truncate = False, vertical = True)

-RECORD 0-------------------------------------------------------------
 events_meta_search_session_id | 0000023E-6AB9-438A-A5C5-24719B03DC8F 
 events_ts                     | 1772889594842                        
 events_meta_keyword           | NULL                                 
 events_meta_action            | search_input                         
 events_meta_content_name      | NULL                                 
 content_id                    | NULL                                 
 events_meta_asset_name        | search_tab                           
 consumption                   | 0                                    
 session                       | 1                                    
 autosuggest                   | 0                                    
-RECORD 1-------------------------------------------------------------
 events_meta_search_session_id | 0000023E-6AB9-438A-A5C5-24719B03DC8F 
 events_ts                     | 1772889596897                        
 event

#

what do I have to do?

I have to train apriori, we have a user searching for mandalorian, obiwan, and rebel, we need to bucket them. like when a user is searching for one thing, we need to find what are the others searches that follow using lift and confidence. It'll be just for exploration, to check what each search correspond to. it needs user and session grouping during training but after it's done, we just need to check the outputs when a keyword is passed. we'll not take the full name, as it will just mean content grouping, it will be on what the partial search results. 

In [4]:
from pyspark.sql.functions import col, when, lit, lower

interactions_data = interactions_data.select("day", "events_ts", "events_meta_action", "events_meta_source_name",\
            "events_av", "events_dt", "events_os", "events_meta_search_session_id", "events_uid", "content_id",\
            "events_appid", "events_meta_keyword", "events_meta_asset_name", "events_meta_content_name", "eventtype")\
            .filter(col("events_meta_search_session_id").isNotNull())\
            .distinct()
path_to_user_mapping = f"gs://data-science-prod-redshift-archive/spectrum/device_user_mapping_optimised/day={date}"
user_mapping_data = spark.read.parquet(path_to_user_mapping).filter(col("temp_uid").isNotNull())\
                    .select("temp_uid", "reg_uid", "day")\
                    .withColumnRenamed("day", "um_day").distinct()
interactions_data1 = interactions_data.join(user_mapping_data,\
                                            on=((user_mapping_data.temp_uid == interactions_data.events_uid) & (user_mapping_data.um_day == interactions_data.day)),\
                                            how="left")

interactions_data1 = interactions_data1.withColumn("uid", when(col("reg_uid").isNull(), col("events_uid")).otherwise(col("reg_uid")))\
                                    .select("day", "events_meta_action","events_ts", "events_meta_source_name",\
                                            "events_av", "events_dt", "events_os", "events_meta_search_session_id", "uid", "content_id",\
                                            "events_appid", "events_meta_keyword", "events_meta_asset_name", "events_meta_content_name", "eventtype")\
                                    .filter((col("events_meta_action").isin(['search_input','text_input', 'voice_search',\
                                                                             'search_click', 'voice search', "search_result_consumed", "tab_click"\
                                                                            'tile_click','banner_cta_click', 'item_search','autosuggest_click'])))

interactions_data1 = interactions_data1.withColumn("consumption",\
                                                   when(((col("events_meta_action").isin(['tile_click','banner_cta_click'])) &\
                                                         col("events_meta_source_name").isin(['search_page','searchLayout']) ) |\
                                                        (col("events_meta_action") == "search_result_consumed"), lit(1))\
                                                   .otherwise(lit(0)))\
                                        .withColumn("session", when(col("events_meta_action")\
                                                        .isin('search_input', 'text_input', 'voice_search', 'search_click', 'voice search') |\
                                                       ((col("events_meta_action") == "tab_click") & (lower("events_meta_asset_name") == 'search') & (lower("events_appid") == 'largescreen'))\
                                                       , lit(1)).otherwise(lit(0))
                                                   )\
                                        .withColumn("autosuggest", when(col("events_meta_action").isin(['item_search','autosuggest_click']) &\
                                                                    col("events_meta_asset_name").isin(['autosuggest','auto_suggestion']) &\
                                                                    (col("eventtype") == "click"), lit(1)).otherwise(lit(0)))

In [13]:
from pyspark.sql import Window
from pyspark.sql import functions as F

# 1. Define the Window specification
# We partition by the ID (grouping) and order by the timestamp
window_spec = Window.partitionBy("events_meta_search_session_id").orderBy("events_ts")

# 2. Apply the window
# If you just want to sort the existing DF, you can use window-based ordering:
final_df = interactions_data1.filter(F.col("session") == 1) \
    .select(
        "events_meta_search_session_id", 
        "events_ts", 
        "events_meta_keyword", 
        "events_meta_action",
        "events_meta_content_name", 
        "content_id", 
        "events_meta_asset_name", 
        "consumption", 
        "session", 
        "autosuggest"
    ) \
    .withColumn("sequence_order", F.row_number().over(window_spec)) \
    .orderBy("events_meta_search_session_id", "events_ts")

# final_df.show(100, truncate=False)

In [7]:
from pyspark.sql import Window
from pyspark.sql import functions as F

# 1. Order DESCENDING so the latest row gets row_number = 1
window_spec_desc = Window.partitionBy("events_meta_search_session_id").orderBy(F.length(F.col("events_meta_search_session_id")).desc())

final_df = interactions_data1.filter(F.col("session") == 1) \
    .select(
        "uid",
        "events_meta_keyword", 
        "events_meta_search_session_id", 
        "events_ts", 
        "events_meta_action",
        "events_meta_content_name", 
        "content_id", 
        "events_meta_asset_name", 
        "consumption", 
        "session", 
        "autosuggest"
    ) \
    .filter(
        col("uid").isNotNull() & 
        col("events_meta_keyword").isNotNull()&
        (~isnan(col("uid"))) & 
        (col("uid") != ""))\
    .withColumn("rn", F.row_number().over(window_spec_desc)) \
    .filter(F.col("rn") == 1) \
    .drop("rn") \
    .orderBy("uid","events_meta_search_session_id", "events_ts")

final_df.show(300, truncate=False)
total_searching_users = final_df.select("uid").distinct().count()

+----------------------------+--------------------------------------------------------------------------+--------------------------------------------------+-------------+------------------+------------------------+----------+----------------------+-----------+-------+-----------+
|uid                         |events_meta_keyword                                                       |events_meta_search_session_id                     |events_ts    |events_meta_action|events_meta_content_name|content_id|events_meta_asset_name|consumption|session|autosuggest|
+----------------------------+--------------------------------------------------------------------------+--------------------------------------------------+-------------+------------------+------------------------+----------+----------------------+-----------+-------+-----------+
|--1jFbhywhnow8Z090          |panc                                                                      |92d201ca-d14e-47bd-919b-730783f5fb11-1772881644896|1

In [5]:
from pyspark.sql import Window
from pyspark.sql import functions as F
from pyspark.sql.functions import col, isnan

# 1. Your existing window for the longest session ID
window_spec_desc = Window.partitionBy("events_meta_search_session_id") \
    .orderBy(F.length(F.col("events_meta_search_session_id")).desc())

# 2. A new window to count occurrences per user
user_window = Window.partitionBy("uid")

multi_searching = interactions_data1.filter(F.col("session") == 1) \
    .select(
        "uid",
        "events_meta_keyword", 
        "events_meta_search_session_id", 
        "events_ts", 
        "events_meta_action",
        "events_meta_content_name", 
        "content_id", 
        "events_meta_asset_name", 
        "consumption", 
        "session", 
        "autosuggest"
    ) \
    .filter(
        col("uid").isNotNull() & 
        col("events_meta_keyword").isNotNull() &
        (~isnan(col("uid"))) & 
        (col("uid") != "")
    )\
    .withColumn("rn", F.row_number().over(window_spec_desc)) \
    .filter(F.col("rn") == 1) \
    .drop("rn") \
    .withColumn("user_record_count", F.count("*").over(user_window)) \
    .filter(F.col("user_record_count") >= 3) \
    .drop("user_record_count") \
    .orderBy("uid", "events_meta_search_session_id", "events_ts")

# multi_searching.show(300, truncate=False)
# multi_searching_users = final_df.select("uid").distinct().count()

In [7]:
multi_searching.select("uid", col("events_meta_keyword").alias("search_key"))\
    .filter(F.length(F.col("search_key"))>3)\
    .distinct().show(300, truncate = False)

+------------------+----------------------------------+
|uid               |search_key                        |
+------------------+----------------------------------+
|-2A15J8MUc9H5J29R0|Barah Aan                         |
|-2A15J8MUc9H5J29R0|tum hi aa                         |
|-2A15J8MUc9H5J29R0|tuta                              |
|-4r1fIimiR46JDQZn0|good wife in tamil                |
|-4r1fIimiR46JDQZn0|geetha govindam in                |
|-4r1fIimiR46JDQZn0|githa govinda                     |
|-5kGOroIAiF2MKhUS0|ambika                            |
|-5kGOroIAiF2MKhUS0|minna                             |
|-5kGOroIAiF2MKhUS0|jean                              |
|-68AGhqQPNkLQF_pW0|pet dect                          |
|-6BBu_QTnErLjwV0n0|LBW: Lov                          |
|-6BBu_QTnErLjwV0n0|Telugu                            |
|-CFJpEbLU8BPT_D-80|naa ninn                          |
|-CFJpEbLU8BPT_D-80|Jagadha                           |
|-CFJpEbLU8BPT_D-80|Naa Ninna Bidalaar          

26/04/24 06:24:05 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 28 for reason Executor for container container_1764236692086_7618_01_000028 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/04/24 06:24:05 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 41 for reason Executor for container container_1764236692086_7618_01_000041 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/04/24 06:24:05 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 163 for reason Executor for container container_1764236692086_7618_01_000165 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/04/24 06:24:05 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 58 for reason Executor for container container_1764236692086_

In [ ]:
print(( /total_searching_users)*100, " percent users multisearch")

11.581819591725957  percent users multisearch


In [19]:
print(total_searching_users)
print(multi_searching_users)

206332
23897


26/04/22 07:27:31 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 95 for reason Executor for container container_1764236692086_7504_01_000097 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/04/22 07:27:31 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 89 for reason Executor for container container_1764236692086_7504_01_000091 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/04/22 07:28:01 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 97 for reason Executor for container container_1764236692086_7504_01_000099 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/04/22 07:28:01 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 101 for reason Executor for container container_1764236692086_

In [ ]:
interactions_data1.select("uid", "events_meta_keyword", "events_meta_action","events_meta_search_session_id", "events_ts","consumption","session","autosuggest","day").filter(
    col("uid").isNotNull() & 
    (~isnan(col("uid"))) & 
    (col("uid") != "")
).orderBy("uid","events_ts").show(500, truncate=False)


from pyspark.sql import Window
from pyspark.sql import functions as F

# 1. Order DESCENDING so the latest row gets row_number = 1
window_spec_desc = Window.partitionBy("events_meta_search_session_id").orderBy(F.col("events_ts").desc())

final_df = interactions_data1.filter(F.col("session") == 1) \
    .select(
        "uid"
        "events_meta_search_session_id", 
        "events_ts", 
        "events_meta_keyword", 
        "events_meta_action",
        "events_meta_content_name", 
        "content_id", 
        "events_meta_asset_name", 
        "consumption", 
        "session", 
        "autosuggest"
    ) \
    .withColumn("rn", F.row_number().over(window_spec_desc)) \
    .filter(F.col("rn") == 1) \
    .drop("rn") \
    .orderBy("uid","events_meta_search_session_id", "events_ts")

final_df.show(20, truncate=False)

+----------------------------+----------------------------------------------------------------+----------------------+--------------------------------------------------+-------------+-----------+-------+-----------+----------+
|uid                         |events_meta_keyword                                             |events_meta_action    |events_meta_search_session_id                     |events_ts    |consumption|session|autosuggest|day       |
+----------------------------+----------------------------------------------------------------+----------------------+--------------------------------------------------+-------------+-----------+-------+-----------+----------+
|--1jFbhywhnow8Z090          |NULL                                                            |text_input            |92d201ca-d14e-47bd-919b-730783f5fb11-1772881644896|1772881645980|0          |1      |0          |2026-03-07|
|--1jFbhywhnow8Z090          |pan                                                           

In [14]:
interactions_data1.count()

3829624

In [ ]:
user_mapping_data.show(20)
user_mapping_data.count()

+--------------------+------------------+----------+
|            temp_uid|           reg_uid|    um_day|
+--------------------+------------------+----------+
|-6LRsCDRs1kg41fkJ...|9RTZ_i7X4IaqDTKUK0|2026-03-07|
|-91k0gR4PWeFizabK...|jOq_TgP47xkFzb0mU0|2026-03-07|
|-IkL17QPvkjtmO_-P...|N0PUCexiNfVc6BHWb0|2026-03-07|
|-M5Qq_yruDjSw7rhS...|VaGrc11xcdOz65FKf0|2026-03-07|
|-NocohqAKGr-xoiJj...|4wTiXuDByyOAkPxqF0|2026-03-07|
|-QZcFQUlHHEH4Lyuy...|zbo0gPws-lfuJaltc0|2026-03-07|
|-R4gqtinT2CpZU0yZ...|ISHfiPcdREAidKcko0|2026-03-07|
|-RN-r6Lk5ieQioq6m...|wPGOAH7Lo_eKcN00S0|2026-03-07|
|-VWQ15UjSngjsBLgW...|CExxAxnJcKnWojkzX0|2026-03-07|
|-bgUUyVEynjs8DqDZ...|4Pf_SKaK_6Y1iagAs0|2026-03-07|
|-cA1_HyA_3OWK4fSn...|qa2AYkias7RRfMvgv0|2026-03-07|
|-eMQKUm30qi_6B7Vb...|DxWyfDBo2zkQgBfxv0|2026-03-07|
|-hkfCmJH_S1xBSlhS...|NAtP3h7WE3cT7EcYr0|2026-03-07|
|-hyJTJfkp_CUrMj4S...|TfJuveoQ2gtyAc_lY0|2026-03-07|
|-i8N8iMY0NOUcr8uy...|H9JaULeAKW8XAP4w50|2026-03-07|
|-jE7lClHUqjR5OvAT...|sHUAhft88k8QpZBZB0|2026-

463211447

# Apriori Pipeline

In [11]:
from pyspark.sql import SparkSession
from pyspark.sql import DataFrame
import pyspark.sql.functions as F
from pyspark.ml.fpm import FPGrowth

class AprioriPipeline:
    """
    A generic PySpark pipeline for Association Rule Mining (Apriori/FP-Growth).
    """
    
    def __init__(self, 
                 user_col: str = "userId", 
                 item_col: str = "contents", 
                 min_support: float = 0.01, 
                 min_confidence: float = 0.1):
        """
        Initializes the pipeline with column names and hyperparameters.
        
        :param user_col: The column representing the transaction ID or user ID.
        :param item_col: The column representing the individual items/contents.
        :param min_support: Minimum support threshold (frequency of itemsets).
        :param min_confidence: Minimum confidence threshold (reliability of rules).
        """
        self.user_col = user_col
        self.item_col = item_col
        self.min_support = min_support
        self.min_confidence = min_confidence
        self.model = None
        
    def preprocess(self, df: DataFrame) -> DataFrame:
        """
        Transforms the flat (userId, content) dataframe into an array of unique 
        items per user, which is the required format for PySpark's FPGrowth.
        """
        # Group by userId and collect unique contents into an array
        transactions_df = df.groupBy(self.user_col) \
                            .agg(F.collect_set(self.item_col).alias("items"))
        return transactions_df

    def fit(self, transactions_df: DataFrame):
        """
        Fits the FP-Growth model on the preprocessed transactions dataframe.
        """
        fp_growth = FPGrowth(
            itemsCol="items", 
            minSupport=self.min_support, 
            minConfidence=self.min_confidence
        )
        self.model = fp_growth.fit(transactions_df)
        print("Model trained successfully.")
        
    def get_frequent_itemsets(self) -> DataFrame:
        """
        Returns the frequent itemsets calculated by the model.
        Returns columns: [items, freq]
        """
        if not self.model:
            raise ValueError("Model is not trained yet. Call .fit() first.")
        return self.model.freqItemsets

    def get_association_rules(self) -> DataFrame:
        """
        Returns the association rules generated by the model.
        Returns columns: [antecedent, consequent, confidence, lift, support]
        """
        if not self.model:
            raise ValueError("Model is not trained yet. Call .fit() first.")
        return self.model.associationRules

    def transform(self, transactions_df: DataFrame) -> DataFrame:
        """
        Recommends new items based on the generated rules.
        Returns columns: [userId, items, prediction]
        """
        if not self.model:
            raise ValueError("Model is not trained yet. Call .fit() first.")
        return self.model.transform(transactions_df)

    def run_pipeline(self, df: DataFrame):
        """
        Master method to run the entire pipeline from raw data to rules.
        """
        print("1. Preprocessing data...")
        transactions = self.preprocess(df)
        
        print("2. Fitting model...")
        self.fit(transactions)
        
        print("3. Fetching rules...")
        rules = self.get_association_rules()
        
        return rules

In [14]:
# 1. Save your transformed data to a DataFrame variable
raw_df = multi_searching.select(
    "uid", 
    F.col("events_meta_keyword").alias("search_key")
).filter(
    F.length(F.col("search_key")) > 3
).distinct()

# 2. Instantiate the pipeline with your specific column names
# Drastically lower the thresholds to force the model to find overlapping patterns
pipeline = AprioriPipeline(
    user_col="uid", 
    item_col="search_key", 
    min_support=0.0005,    # Lowered from 0.01
    min_confidence=0.01   # Lowered from 0.1
)

rules_df = pipeline.run_pipeline(raw_df)

# 4. Display the resulting association rules, ordered by lift (strength of association)
print("Generated Association Rules:")
rules_df.orderBy(F.desc("lift")).show(truncate=False)

# Optional: If you want to see the frequent itemsets as well
frequent_itemsets = pipeline.get_frequent_itemsets()
frequent_itemsets.orderBy(F.desc("freq")).show(truncate=False)

1. Preprocessing data...
2. Fitting model...


Model trained successfully.
3. Fetching rules...
Generated Association Rules:


+-------------------------+-------------------------+-------------------+------------------+---------------------+
|antecedent               |consequent               |confidence         |lift              |support              |
+-------------------------+-------------------------+-------------------+------------------+---------------------+
|[mont]                   |[montu]                  |0.36363636363636365|170.3111111111111 |5.693680015183147E-4 |
|[montu]                  |[mont]                   |0.26666666666666666|170.3111111111111 |5.693680015183147E-4 |
|[montu pilot]            |[mont]                   |0.23076923076923078|147.3846153846154 |5.693680015183147E-4 |
|[mont]                   |[montu pilot]            |0.36363636363636365|147.3846153846154 |5.693680015183147E-4 |
|[montu pilot]            |[montu]                  |0.25               |117.08888888888887|6.168153349781743E-4 |
|[montu]                  |[montu pilot]            |0.28888888888888886|117.088

In [15]:
rules_df.orderBy(F.desc("lift")).show(500, truncate=False)


+----------------------------------------------------+----------------------------------------+--------------------+------------------+---------------------+
|antecedent                                          |consequent                              |confidence          |lift              |support              |
+----------------------------------------------------+----------------------------------------+--------------------+------------------+---------------------+
|[mont]                                              |[montu]                                 |0.36363636363636365 |170.3111111111111 |5.693680015183147E-4 |
|[montu]                                             |[mont]                                  |0.26666666666666666 |170.3111111111111 |5.693680015183147E-4 |
|[montu pilot]                                       |[mont]                                  |0.23076923076923078 |147.3846153846154 |5.693680015183147E-4 |
|[mont]                                             

26/04/24 10:17:24 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 141 for reason Executor for container container_1764236692086_7652_01_000146 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/04/24 10:17:24 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 134 for reason Executor for container container_1764236692086_7652_01_000137 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/04/24 10:17:45 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 127 for reason Executor for container container_1764236692086_7652_01_000130 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/04/24 10:17:45 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 151 for reason Executor for container container_17642366920